# Phase 4.2: Hyperparameter Tuning
## Credit Scoring - Give Me Some Credit

**Author**: QuangMinh  
**Course**: MLE501 - AI & Machine Learning  
**Date**: 2026-03-13

---

## Mục tiêu

Tuning nhẹ (RandomizedSearchCV) cho **2 models tốt nhất** từ Phase 3:
1. **LightGBM** (AUC = 0.8672, Rank #1)
2. **XGBoost** (AUC = 0.8506, Rank #2)

### So với Phase 3 (default params):
- Phase 3: Train 5 models với default parameters
- **Phase 4.2**: Tuning LightGBM + XGBoost với RandomizedSearchCV
- Kết quả lưu **riêng** để so sánh trước/sau tuning

### Cấu hình nhẹ (tránh đơ máy):
- `RandomizedSearchCV` (không dùng GridSearchCV)
- `n_iter=20` (thử 20 combinations ngẫu nhiên)
- `cv=3` (3-fold cross-validation)
- `n_jobs=2` (dùng 2 CPU cores, không dùng -1)
- Thời gian ước tính: **2-6 phút tổng cộng**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, precision_score,
    classification_report, confusion_matrix, roc_curve
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

from datetime import datetime
print(f"Phase 4.2 started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Output directory — lưu riêng so với Phase 3/4
report_dir = '../reports/[4.2] Ket qua Tuning/'
os.makedirs(report_dir, exist_ok=True)
print("Libraries imported successfully!")

## 2. Load Data

In [ ]:
print("="*80)
print("LOADING DATA")
print("="*80)

data_dir = '../data/processed/'

# Unscaled data (for tree-based models)
X_train = pd.read_csv(data_dir + 'X_train.csv')
X_val = pd.read_csv(data_dir + 'X_val.csv')
y_train = pd.read_csv(data_dir + 'y_train.csv').squeeze()
y_val = pd.read_csv(data_dir + 'y_val.csv').squeeze()

# Imbalance ratio
imbalance_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

print(f"\nTraining:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
print(f"Features: {X_train.columns.tolist()}")

## 3. Kết quả Default (Phase 3) — Baseline để so sánh

In [ ]:
print("="*80)
print("KẾT QUẢ DEFAULT PARAMS (Phase 3) — Baseline")
print("="*80)

# Load Phase 3 results
phase3_results = pd.read_csv('../reports/[3] Ket qua Model Building/model_comparison.csv')
print()
display(phase3_results.style.format({
    'AUC-ROC': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'Precision': '{:.4f}',
    'Train Time (s)': '{:.2f}'
}))

# Lưu baseline metrics
lgbm_default = phase3_results[phase3_results['Model'] == 'LightGBM'].iloc[0]
xgb_default = phase3_results[phase3_results['Model'] == 'XGBoost'].iloc[0]

print(f"\nLightGBM default: AUC={lgbm_default['AUC-ROC']:.4f}, Recall={lgbm_default['Recall']:.4f}")
print(f"XGBoost  default: AUC={xgb_default['AUC-ROC']:.4f}, Recall={xgb_default['Recall']:.4f}")

## 4. Tuning LightGBM

### Default params (Phase 3):
- n_estimators=100, num_leaves=31, learning_rate=0.1, is_unbalance=True

### Param search space:

In [ ]:
print("="*80)
print("TUNING LightGBM — RandomizedSearchCV")
print("="*80)

lgbm_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 63],
    'max_depth': [3, 5, 7, -1],
    'min_child_samples': [20, 50, 100],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

total_combinations = 1
for v in lgbm_param_dist.values():
    total_combinations *= len(v)
print(f"\nTổng combinations có thể: {total_combinations}")
print(f"Số combinations sẽ thử: 20 (RandomizedSearch)")
print(f"CV folds: 3")
print(f"Tổng lần train: 20 × 3 = 60")

# Cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

lgbm_search = RandomizedSearchCV(
    estimator=LGBMClassifier(
        is_unbalance=True,
        random_state=42,
        verbose=-1,
        n_jobs=1  # 1 job per model (outer parallelism handled by n_jobs=2)
    ),
    param_distributions=lgbm_param_dist,
    n_iter=20,
    cv=cv_strategy,
    scoring='roc_auc',
    random_state=42,
    n_jobs=2,       # Dùng 2 cores — tránh đơ máy
    verbose=1,
    return_train_score=True
)

print("\nBắt đầu tuning...")
start_time = time.time()
lgbm_search.fit(X_train, y_train)
lgbm_tuning_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"KẾT QUẢ TUNING LightGBM")
print(f"{'='*80}")
print(f"Thời gian tuning: {lgbm_tuning_time:.1f}s ({lgbm_tuning_time/60:.1f} phút)")
print(f"Best AUC-ROC (CV): {lgbm_search.best_score_:.4f}")
print(f"\nBest params:")
for param, value in lgbm_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# Đánh giá trên Validation set
lgbm_tuned = lgbm_search.best_estimator_
lgbm_tuned_pred = lgbm_tuned.predict(X_val)
lgbm_tuned_prob = lgbm_tuned.predict_proba(X_val)[:, 1]

lgbm_tuned_metrics = {
    'AUC-ROC': roc_auc_score(y_val, lgbm_tuned_prob),
    'Recall': recall_score(y_val, lgbm_tuned_pred),
    'F1-Score': f1_score(y_val, lgbm_tuned_pred),
    'Precision': precision_score(y_val, lgbm_tuned_pred)
}

print("="*80)
print("LightGBM TUNED — Validation Set")
print("="*80)
print(f"\n  AUC-ROC:   {lgbm_tuned_metrics['AUC-ROC']:.4f}  (default: {lgbm_default['AUC-ROC']:.4f}, change: {lgbm_tuned_metrics['AUC-ROC'] - lgbm_default['AUC-ROC']:+.4f})")
print(f"  Recall:    {lgbm_tuned_metrics['Recall']:.4f}  (default: {lgbm_default['Recall']:.4f}, change: {lgbm_tuned_metrics['Recall'] - lgbm_default['Recall']:+.4f})")
print(f"  F1-Score:  {lgbm_tuned_metrics['F1-Score']:.4f}  (default: {lgbm_default['F1-Score']:.4f}, change: {lgbm_tuned_metrics['F1-Score'] - lgbm_default['F1-Score']:+.4f})")
print(f"  Precision: {lgbm_tuned_metrics['Precision']:.4f}  (default: {lgbm_default['Precision']:.4f}, change: {lgbm_tuned_metrics['Precision'] - lgbm_default['Precision']:+.4f})")

print(f"\nClassification Report:")
print(classification_report(y_val, lgbm_tuned_pred, target_names=['Good (0)', 'Bad (1)'], digits=4))

cm = confusion_matrix(y_val, lgbm_tuned_pred)
print(f"Confusion Matrix:")
print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")

## 5. Tuning XGBoost

### Default params (Phase 3):
- n_estimators=100, max_depth=6, learning_rate=0.3, scale_pos_weight=13.96

### Param search space:

In [ ]:
print("="*80)
print("TUNING XGBoost — RandomizedSearchCV")
print("="*80)

xgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1, 0.5]
}

total_combinations_xgb = 1
for v in xgb_param_dist.values():
    total_combinations_xgb *= len(v)
print(f"\nTổng combinations có thể: {total_combinations_xgb}")
print(f"Số combinations sẽ thử: 20 (RandomizedSearch)")
print(f"CV folds: 3")
print(f"Tổng lần train: 20 × 3 = 60")

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        scale_pos_weight=imbalance_ratio,
        eval_metric='logloss',
        random_state=42,
        n_jobs=1
    ),
    param_distributions=xgb_param_dist,
    n_iter=20,
    cv=cv_strategy,
    scoring='roc_auc',
    random_state=42,
    n_jobs=2,       # Dùng 2 cores — tránh đơ máy
    verbose=1,
    return_train_score=True
)

print("\nBắt đầu tuning...")
start_time = time.time()
xgb_search.fit(X_train, y_train)
xgb_tuning_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"KẾT QUẢ TUNING XGBoost")
print(f"{'='*80}")
print(f"Thời gian tuning: {xgb_tuning_time:.1f}s ({xgb_tuning_time/60:.1f} phút)")
print(f"Best AUC-ROC (CV): {xgb_search.best_score_:.4f}")
print(f"\nBest params:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# Đánh giá trên Validation set
xgb_tuned = xgb_search.best_estimator_
xgb_tuned_pred = xgb_tuned.predict(X_val)
xgb_tuned_prob = xgb_tuned.predict_proba(X_val)[:, 1]

xgb_tuned_metrics = {
    'AUC-ROC': roc_auc_score(y_val, xgb_tuned_prob),
    'Recall': recall_score(y_val, xgb_tuned_pred),
    'F1-Score': f1_score(y_val, xgb_tuned_pred),
    'Precision': precision_score(y_val, xgb_tuned_pred)
}

print("="*80)
print("XGBoost TUNED — Validation Set")
print("="*80)
print(f"\n  AUC-ROC:   {xgb_tuned_metrics['AUC-ROC']:.4f}  (default: {xgb_default['AUC-ROC']:.4f}, change: {xgb_tuned_metrics['AUC-ROC'] - xgb_default['AUC-ROC']:+.4f})")
print(f"  Recall:    {xgb_tuned_metrics['Recall']:.4f}  (default: {xgb_default['Recall']:.4f}, change: {xgb_tuned_metrics['Recall'] - xgb_default['Recall']:+.4f})")
print(f"  F1-Score:  {xgb_tuned_metrics['F1-Score']:.4f}  (default: {xgb_default['F1-Score']:.4f}, change: {xgb_tuned_metrics['F1-Score'] - xgb_default['F1-Score']:+.4f})")
print(f"  Precision: {xgb_tuned_metrics['Precision']:.4f}  (default: {xgb_default['Precision']:.4f}, change: {xgb_tuned_metrics['Precision'] - xgb_default['Precision']:+.4f})")

print(f"\nClassification Report:")
print(classification_report(y_val, xgb_tuned_pred, target_names=['Good (0)', 'Bad (1)'], digits=4))

cm = confusion_matrix(y_val, xgb_tuned_pred)
print(f"Confusion Matrix:")
print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")

## 6. So sánh: Default vs Tuned vs Tất cả 5 Models

In [ ]:
print("="*80)
print("SO SÁNH TỔNG HỢP: 5 Models (Default) + 2 Models (Tuned)")
print("="*80)

# Load tất cả kết quả Phase 3
all_results = []

for _, row in phase3_results.iterrows():
    all_results.append({
        'Model': row['Model'],
        'Version': 'Default',
        'AUC-ROC': row['AUC-ROC'],
        'Recall': row['Recall'],
        'F1-Score': row['F1-Score'],
        'Precision': row['Precision']
    })

# Thêm kết quả tuned
all_results.append({
    'Model': 'LightGBM',
    'Version': 'Tuned',
    'AUC-ROC': lgbm_tuned_metrics['AUC-ROC'],
    'Recall': lgbm_tuned_metrics['Recall'],
    'F1-Score': lgbm_tuned_metrics['F1-Score'],
    'Precision': lgbm_tuned_metrics['Precision']
})

all_results.append({
    'Model': 'XGBoost',
    'Version': 'Tuned',
    'AUC-ROC': xgb_tuned_metrics['AUC-ROC'],
    'Recall': xgb_tuned_metrics['Recall'],
    'F1-Score': xgb_tuned_metrics['F1-Score'],
    'Precision': xgb_tuned_metrics['Precision']
})

comparison_df = pd.DataFrame(all_results).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
comparison_df.index += 1

print()
display(comparison_df.style.format({
    'AUC-ROC': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'Precision': '{:.4f}'
}).apply(lambda x: ['background-color: #d4edda' if v == 'Tuned' else '' for v in x], subset=['Version']))

In [ ]:
# Bảng so sánh trực tiếp Default vs Tuned
print("="*80)
print("SO SÁNH TRỰC TIẾP: Default vs Tuned")
print("="*80)

direct_comparison = pd.DataFrame([
    {
        'Model': 'LightGBM',
        'AUC_Default': lgbm_default['AUC-ROC'],
        'AUC_Tuned': lgbm_tuned_metrics['AUC-ROC'],
        'AUC_Change': lgbm_tuned_metrics['AUC-ROC'] - lgbm_default['AUC-ROC'],
        'Recall_Default': lgbm_default['Recall'],
        'Recall_Tuned': lgbm_tuned_metrics['Recall'],
        'Recall_Change': lgbm_tuned_metrics['Recall'] - lgbm_default['Recall'],
        'F1_Default': lgbm_default['F1-Score'],
        'F1_Tuned': lgbm_tuned_metrics['F1-Score'],
        'F1_Change': lgbm_tuned_metrics['F1-Score'] - lgbm_default['F1-Score'],
    },
    {
        'Model': 'XGBoost',
        'AUC_Default': xgb_default['AUC-ROC'],
        'AUC_Tuned': xgb_tuned_metrics['AUC-ROC'],
        'AUC_Change': xgb_tuned_metrics['AUC-ROC'] - xgb_default['AUC-ROC'],
        'Recall_Default': xgb_default['Recall'],
        'Recall_Tuned': xgb_tuned_metrics['Recall'],
        'Recall_Change': xgb_tuned_metrics['Recall'] - xgb_default['Recall'],
        'F1_Default': xgb_default['F1-Score'],
        'F1_Tuned': xgb_tuned_metrics['F1-Score'],
        'F1_Change': xgb_tuned_metrics['F1-Score'] - xgb_default['F1-Score'],
    }
])

print()
display(direct_comparison.style.format({
    'AUC_Default': '{:.4f}', 'AUC_Tuned': '{:.4f}', 'AUC_Change': '{:+.4f}',
    'Recall_Default': '{:.4f}', 'Recall_Tuned': '{:.4f}', 'Recall_Change': '{:+.4f}',
    'F1_Default': '{:.4f}', 'F1_Tuned': '{:.4f}', 'F1_Change': '{:+.4f}'
}))

## 7. Visualization

In [ ]:
# ROC Curves: Default vs Tuned
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Load default models
lgbm_default_model = joblib.load('../models/lightgbm.pkl')
xgb_default_model = joblib.load('../models/xgboost.pkl')

# --- LightGBM ---
ax = axes[0]
# Default
fpr_d, tpr_d, _ = roc_curve(y_val, lgbm_default_model.predict_proba(X_val)[:, 1])
auc_d = roc_auc_score(y_val, lgbm_default_model.predict_proba(X_val)[:, 1])
ax.plot(fpr_d, tpr_d, 'b--', lw=2, label=f'Default (AUC={auc_d:.4f})')
# Tuned
fpr_t, tpr_t, _ = roc_curve(y_val, lgbm_tuned_prob)
auc_t = lgbm_tuned_metrics['AUC-ROC']
ax.plot(fpr_t, tpr_t, 'r-', lw=2, label=f'Tuned (AUC={auc_t:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
ax.set_title('LightGBM: Default vs Tuned', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# --- XGBoost ---
ax = axes[1]
# Default
fpr_d, tpr_d, _ = roc_curve(y_val, xgb_default_model.predict_proba(X_val)[:, 1])
auc_d = roc_auc_score(y_val, xgb_default_model.predict_proba(X_val)[:, 1])
ax.plot(fpr_d, tpr_d, 'b--', lw=2, label=f'Default (AUC={auc_d:.4f})')
# Tuned
fpr_t, tpr_t, _ = roc_curve(y_val, xgb_tuned_prob)
auc_t = xgb_tuned_metrics['AUC-ROC']
ax.plot(fpr_t, tpr_t, 'r-', lw=2, label=f'Tuned (AUC={auc_t:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
ax.set_title('XGBoost: Default vs Tuned', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.suptitle('ROC Curves — Default vs Tuned', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(report_dir + 'roc_default_vs_tuned.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: roc_default_vs_tuned.png")

In [ ]:
# Bar chart: Default vs Tuned metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_names = ['AUC-ROC', 'Recall', 'F1-Score']
x = np.arange(2)
width = 0.35

default_vals = {
    'AUC-ROC': [lgbm_default['AUC-ROC'], xgb_default['AUC-ROC']],
    'Recall': [lgbm_default['Recall'], xgb_default['Recall']],
    'F1-Score': [lgbm_default['F1-Score'], xgb_default['F1-Score']]
}

tuned_vals = {
    'AUC-ROC': [lgbm_tuned_metrics['AUC-ROC'], xgb_tuned_metrics['AUC-ROC']],
    'Recall': [lgbm_tuned_metrics['Recall'], xgb_tuned_metrics['Recall']],
    'F1-Score': [lgbm_tuned_metrics['F1-Score'], xgb_tuned_metrics['F1-Score']]
}

for i, metric in enumerate(metrics_names):
    ax = axes[i]
    bars1 = ax.bar(x - width/2, default_vals[metric], width, label='Default', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x + width/2, tuned_vals[metric], width, label='Tuned', color='#e74c3c', alpha=0.8)
    
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['LightGBM', 'XGBoost'])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Value labels
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
    
    if metric == 'AUC-ROC':
        ax.set_ylim(0.8, 0.95)
    else:
        ax.set_ylim(0, 1)

plt.suptitle('Default vs Tuned — Metrics Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(report_dir + 'metrics_default_vs_tuned.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: metrics_default_vs_tuned.png")

## 8. Save Tuned Models & Results

In [ ]:
print("="*80)
print("SAVING TUNED MODELS & RESULTS")
print("="*80)

# Save tuned models (riêng so với default models)
models_dir = '../models/'
joblib.dump(lgbm_tuned, models_dir + 'lightgbm_tuned.pkl')
joblib.dump(xgb_tuned, models_dir + 'xgboost_tuned.pkl')
print(f"\nModels saved:")
print(f"  - {models_dir}lightgbm_tuned.pkl")
print(f"  - {models_dir}xgboost_tuned.pkl")

# Save comparison table
comparison_df.to_csv(report_dir + 'tuning_comparison.csv', index=False)
print(f"  - {report_dir}tuning_comparison.csv")

# Save direct comparison
direct_comparison.to_csv(report_dir + 'default_vs_tuned.csv', index=False)
print(f"  - {report_dir}default_vs_tuned.csv")

# Save best params
best_params = pd.DataFrame([
    {'Model': 'LightGBM', **lgbm_search.best_params_, 'Best_CV_AUC': lgbm_search.best_score_},
    {'Model': 'XGBoost', **xgb_search.best_params_, 'Best_CV_AUC': xgb_search.best_score_}
])
best_params.to_csv(report_dir + 'best_params.csv', index=False)
print(f"  - {report_dir}best_params.csv")

print(f"\n{'='*80}")
print("ALL FILES SAVED SUCCESSFULLY!")
print(f"{'='*80}")

## 9. Kết luận

In [ ]:
print("="*80)
print("KẾT LUẬN PHASE 4.2 — HYPERPARAMETER TUNING")
print("="*80)

print(f"""
1. TUNING CONFIG:
   - Method: RandomizedSearchCV (n_iter=20, cv=3)
   - LightGBM tuning time: {lgbm_tuning_time:.1f}s
   - XGBoost  tuning time: {xgb_tuning_time:.1f}s
   - Total: {lgbm_tuning_time + xgb_tuning_time:.1f}s ({(lgbm_tuning_time + xgb_tuning_time)/60:.1f} phut)

2. KET QUA:

   LightGBM:
     AUC-ROC: {lgbm_default['AUC-ROC']:.4f} → {lgbm_tuned_metrics['AUC-ROC']:.4f} ({lgbm_tuned_metrics['AUC-ROC'] - lgbm_default['AUC-ROC']:+.4f})
     Recall:  {lgbm_default['Recall']:.4f} → {lgbm_tuned_metrics['Recall']:.4f} ({lgbm_tuned_metrics['Recall'] - lgbm_default['Recall']:+.4f})
     F1:      {lgbm_default['F1-Score']:.4f} → {lgbm_tuned_metrics['F1-Score']:.4f} ({lgbm_tuned_metrics['F1-Score'] - lgbm_default['F1-Score']:+.4f})

   XGBoost:
     AUC-ROC: {xgb_default['AUC-ROC']:.4f} → {xgb_tuned_metrics['AUC-ROC']:.4f} ({xgb_tuned_metrics['AUC-ROC'] - xgb_default['AUC-ROC']:+.4f})
     Recall:  {xgb_default['Recall']:.4f} → {xgb_tuned_metrics['Recall']:.4f} ({xgb_tuned_metrics['Recall'] - xgb_default['Recall']:+.4f})
     F1:      {xgb_default['F1-Score']:.4f} → {xgb_tuned_metrics['F1-Score']:.4f} ({xgb_tuned_metrics['F1-Score'] - xgb_default['F1-Score']:+.4f})

3. BEST PARAMS:
""")

print("   LightGBM:")
for param, value in lgbm_search.best_params_.items():
    print(f"     {param}: {value}")

print("\n   XGBoost:")
for param, value in xgb_search.best_params_.items():
    print(f"     {param}: {value}")

print(f"\n4. FILES SAVED:")
print(f"   - models/lightgbm_tuned.pkl")
print(f"   - models/xgboost_tuned.pkl")
print(f"   - reports/[4.2] Ket qua Tuning/tuning_comparison.csv")
print(f"   - reports/[4.2] Ket qua Tuning/default_vs_tuned.csv")
print(f"   - reports/[4.2] Ket qua Tuning/best_params.csv")
print(f"   - reports/[4.2] Ket qua Tuning/roc_default_vs_tuned.png")
print(f"   - reports/[4.2] Ket qua Tuning/metrics_default_vs_tuned.png")

print(f"\nPhase 4.2 completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")